In [1]:
import pandas as pd
from pathlib import Path


def global_stats_multiindex(run_df: pd.DataFrame, filter_empty: bool = False) -> pd.DataFrame:
    """
    Compute global mean & std of all metrics using a MultiIndex column layout.
    If filter_empty=True, excludes rows where the corresponding *_length is 0.
    """
    metric_cols = [
        "forbidden_length",
        "forbidden_Precision",
        "forbidden_Recall",
        "forbidden_F1",
        "required_length",
        "required_Precision",
        "required_Recall",
        "required_F1",
    ]

    means, stds = {}, {}

    if filter_empty:
        # Filtered version — skip rows where *_length == 0
        forbidden_df = run_df[run_df["forbidden_length"] > 0]
        for col in ["forbidden_length", "forbidden_Precision", "forbidden_Recall", "forbidden_F1"]:
            means[col] = forbidden_df[col].mean()
            stds[col] = forbidden_df[col].std()

        required_df = run_df[run_df["required_length"] > 0]
        for col in ["required_length", "required_Precision", "required_Recall", "required_F1"]:
            means[col] = required_df[col].mean()
            stds[col] = required_df[col].std()
    else:
        # Unfiltered — use all rows
        for col in metric_cols:
            means[col] = run_df[col].mean()
            stds[col] = run_df[col].std()

    means_df = pd.DataFrame([means])
    stds_df = pd.DataFrame([stds])

    combined = pd.concat([means_df, stds_df], axis=1, keys=["mean", "std"])
    combined = combined.swaplevel(0, 1, axis=1)
    combined = combined.reindex(
        sorted(combined.columns, key=lambda t: metric_cols.index(t[0])), axis=1
    )

    return combined


def merge_all_global_stats(results_dir: str, filter_empty: bool = False) -> pd.DataFrame:
    """
    Loads all predefined result files from results_dir, computes global statistics
    (filtered or unfiltered), and merges them into a single MultiIndex DataFrame.
    """

    # Hardcoded filenames in consistent order
    filenames = [
        "bnlearn.json",
        "bnlearn-desc.json",
        "bnlearn-consensus.json",
        "bnlearn-desc-consensus.json",
        "synthetic.json",
        "synthetic-desc.json",
        "synthetic-consensus.json",
        "synthetic-desc-consensus.json",
    ]

    results = []
    for fname in filenames:
        path = Path(results_dir) / fname
        if not path.exists():
            print(f"⚠️ Warning: {fname} not found, skipping.")
            continue

        try:
            df = pd.read_json(path)
            stats = global_stats_multiindex(df, filter_empty=filter_empty)
            stats.insert(0, "source", Path(fname).stem)
            results.append(stats)
        except Exception as e:
            print(f"⚠️ Error processing {fname}: {e}")

    merged = pd.concat(results, axis=0, ignore_index=True)
    merged = merged.set_index("source")

    return merged

In [2]:
merged_unfiltered = merge_all_global_stats("../results/llm_constraints")
merged_unfiltered

forbidden_length            forbidden_Precision  \
                                     mean        std                mean   
source                                                                     
bnlearn                         19.800000  13.902121            0.927031   
bnlearn-desc                    25.466667  24.387296            0.973218   
bnlearn-consensus                8.000000   6.603030            0.833333   
bnlearn-desc-consensus           8.166667   1.722401            0.948148   
synthetic                       17.592593  13.020885            0.952067   
synthetic-desc                  20.640741  16.603864            0.941930   
synthetic-consensus              4.851852   4.834652            0.854938   
synthetic-desc-consensus         4.074074   3.424947            0.876852   

                                   forbidden_Recall           forbidden_F1  \
                               std             mean       std         mean   
source                                                                       
bnlearn                   0.185501         0.408824  0.234231     0.532704   
bnlearn-desc              0.040794         0.471636  0.218569     0.606145   
bnlearn-consensus         0.408248         0.241056  0.239977     0.337822   
bnlearn-desc-consensus    0.085105         0.274813  0.215987     0.391534   
synthetic                 0.131790         0.292808  0.216026     0.409001   
synthetic-desc            0.154712         0.311809  0.220745     0.430799   
synthetic-consensus       0.339250         0.123099  0.164026     0.188528   
synthetic-desc-consensus  0.316930         0.114940  0.167522     0.174197   

                                   required_length             \
                               std            mean        std   
source                                                          
bnlearn                   0.260416        8.266667   6.967577   
bnlearn-desc              0.224026       10.800000  11.713240   
bnlearn-consensus         0.315565        4.500000   3.728270   
bnlearn-desc-consensus    0.280756        3.666667   4.131182   
synthetic                 0.231825       12.570370   9.597119   
synthetic-desc            0.233594       14.555556  11.524770   
synthetic-consensus       0.215794        4.111111   3.294745   
synthetic-desc-consensus  0.218401        4.740741   3.331866   

                         required_Precision           required_Recall  \
                                       mean       std            mean   
source                                                                  
bnlearn                            0.468148  0.287821        0.445451   
bnlearn-desc                       0.597090  0.235115        0.610918   
bnlearn-consensus                  0.374868  0.327216        0.302941   
bnlearn-desc-consensus             0.504545  0.422263        0.281275   
synthetic                          0.487247  0.249399        0.486291   
synthetic-desc                     0.464399  0.229875        0.515804   
synthetic-consensus                0.568024  0.353107        0.243766   
synthetic-desc-consensus           0.514066  0.334913        0.277768   

                                   required_F1            
                               std        mean       std  
source                                                    
bnlearn                   0.351526    0.401306  0.240443  
bnlearn-desc              0.327527    0.555144  0.232414  
bnlearn-consensus         0.388124    0.283710  0.284446  
bnlearn-desc-consensus    0.375246    0.320635  0.336777  
synthetic                 0.254619    0.450360  0.206281  
synthetic-desc            0.242043    0.453372  0.190414  
synthetic-consensus       0.209589    0.314296  0.227530  
synthetic-desc-consensus  0.243767    0.334160  0.248860

In [3]:
merged_filtered = merge_all_global_stats("../results/llm_constraints", filter_empty=True)
merged_filtered

forbidden_length            forbidden_Precision  \
                                     mean        std                mean   
source                                                                     
bnlearn                         20.482759  13.626709            0.958998   
bnlearn-desc                    25.466667  24.387296            0.973218   
bnlearn-consensus                9.600000   5.941380            1.000000   
bnlearn-desc-consensus           8.166667   1.722401            0.948148   
synthetic                       17.857143  12.936794            0.966384   
synthetic-desc                  20.794776  16.569196            0.948959   
synthetic-consensus              5.574468   4.776504            0.982270   
synthetic-desc-consensus         4.583333   3.293019            0.986458   

                                   forbidden_Recall           forbidden_F1  \
                               std             mean       std         mean   
source                                                                       
bnlearn                   0.062360         0.422922  0.225052     0.551073   
bnlearn-desc              0.040794         0.471636  0.218569     0.606145   
bnlearn-consensus         0.000000         0.289267  0.233568     0.405386   
bnlearn-desc-consensus    0.085105         0.274813  0.215987     0.391534   
synthetic                 0.061180         0.297211  0.214611     0.415151   
synthetic-desc            0.131983         0.314136  0.219908     0.434014   
synthetic-consensus       0.069366         0.141433  0.168372     0.216607   
synthetic-desc-consensus  0.053335         0.129307  0.172479     0.195971   

                                   required_length             \
                               std            mean        std   
source                                                          
bnlearn                   0.244447        9.920000   6.447997   
bnlearn-desc              0.224026       11.172414  11.738426   
bnlearn-consensus         0.300398        6.750000   1.707825   
bnlearn-desc-consensus    0.280756        5.500000   3.872983   
synthetic                 0.228016       12.664179   9.570902   
synthetic-desc            0.231462       14.664179  11.498539   
synthetic-consensus       0.217805        4.530612   3.169530   
synthetic-desc-consensus  0.222330        5.019608   3.215526   

                         required_Precision           required_Recall  \
                                       mean       std            mean   
source                                                                  
bnlearn                            0.561778  0.212859        0.534541   
bnlearn-desc                       0.617680  0.209956        0.631984   
bnlearn-consensus                  0.562302  0.194742        0.454412   
bnlearn-desc-consensus             0.756818  0.206422        0.421912   
synthetic                          0.490883  0.246726        0.489920   
synthetic-desc                     0.467864  0.227180        0.519653   
synthetic-consensus                0.625985  0.317228        0.268640   
synthetic-desc-consensus           0.544306  0.319543        0.294107   

                                   required_F1            
                               std        mean       std  
source                                                    
bnlearn                   0.315766    0.481568  0.172034  
bnlearn-desc              0.311955    0.574286  0.211091  
bnlearn-consensus         0.399116    0.425565  0.233147  
bnlearn-desc-consensus    0.394421    0.480952  0.293641  
synthetic                 0.252055    0.453720  0.203322  
synthetic-desc            0.238780    0.456756  0.187024  
synthetic-consensus       0.204162    0.346367  0.214063  
synthetic-desc-consensus  0.241011    0.353817  0.241978